# 06 · Module 3 checkpoint: a reproducible Table 1 and three figures

**Module:** 3 · **Time:** 60 minutes · **Prerequisites:** Modules 2 and 3 lessons 01–05

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/03_describe_visualize/06_checkpoint.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F03_describe_visualize%2F06_checkpoint.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** These badges target `main` after merge; the repository is currently private.

## Learning goals

- **Clean** an illustrative survey wage variable before reporting statistics.
- **Build** a descriptive table with transparent counts and means.
- **Produce** three clearly labeled figures from one reproducible workflow.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(20261101)
regions = ["North", "SOUTH", "east", "West"]
statuses = ["employed", "unemployed", "not in labor force"]
print("Synthetic data seed:", 20261101)

## Why this matters

A research report should use the same cleaned analysis sample for its table and figures. This checkpoint regenerates the deterministic fictional survey used in the Module 2 project, without depending on network access. Every result below is an exercise in transparent data handling, **not** evidence about real households.

## Recreate the fictional survey

The generator mirrors the 90-record project recipe with a fixed random seed. It deliberately adds missing codes, text values, and one extreme wage to illustrate what must be checked before plotting.

In [ ]:
def make_response(i):
    wage = int(rng.integers(12000, 88000))
    return {
        "person_id": 1001 + i, "household_id": f"H{i // 3 + 1:03d}",
        "region": regions[i % len(regions)],
        "employment": statuses[i % len(statuses)],
        "wage_2023": str(wage),
        "wage_2024": str(round(wage * float(rng.uniform(0.9, 1.2)))),
        "survey_weight": round(float(rng.uniform(0.6, 2.4)), 3),
    }

In [ ]:
rows = [make_response(i) for i in range(90)]
rows[0]["wage_2023"] = "-99"
rows[1]["wage_2024"] = "not reported"
rows[2]["region"] = " north "
rows[3]["wage_2024"] = "9999999"
rows[5]["person_id"] = rows[4]["person_id"]
survey = pd.DataFrame(rows)
print("Raw responses:", len(survey))

## Inspect before analysis

Do not treat the missing-value code or the implausibly large wage as a legitimate observation. Keep the original columns intact for auditability.

In [ ]:
print("Duplicate IDs:", survey["person_id"].duplicated().sum())
print("2024 text values:", survey["wage_2024"].head(4).tolist())
print("Regions:", sorted(survey["region"].unique()))

## If you know Stata/R

| Task | Python | Stata | R |
| --- | --- | --- | --- |
| Recode missing | `pd.to_numeric(...,errors='coerce')` | `destring` plus `replace` | `readr::parse_number()` plus `na_if()` |
| Group summary | `groupby().agg()` | `collapse` | `group_by() |> summarise()` |
| Histogram | `ax.hist(...)` | `histogram wage` | `geom_histogram()` |
| Group bar chart | `ax.bar(...)` | `graph bar` | `geom_col()` |

All three charts should include **units** and make the fictional source explicit.

## Try it

This checkpoint is intentionally less guided than earlier lessons. Complete each answer before comparing your work with the matching solution notebook.

### ★ Exercise 1 — clean the 2024 wage

Create `clean_wage24`, converting strings to numeric and treating wages above 500,000 as invalid for this teaching exercise.

In [ ]:
clean_wage24 = None

In [ ]:
expected_wage24 = pd.to_numeric(survey["wage_2024"], errors="coerce")
expected_wage24 = expected_wage24.mask(expected_wage24 > 500_000)
if (isinstance(clean_wage24, pd.Series) and
    np.allclose(clean_wage24, expected_wage24, equal_nan=True)):
    print("✓ Cleaned wage values correct")
else:
    print("Hint: convert with errors='coerce' and mask values above 500000.")

### ★★ Exercise 2 — build a descriptive Table 1

Build `table1` indexed by normalized region (`str.strip().str.lower()`), with columns `n_wages` and `mean_wage` for your cleaned 2024 variable.

In [ ]:
table1 = None

In [ ]:
expected_region = survey["region"].str.strip().str.lower()
expected_table = pd.DataFrame({"region": expected_region,
                               "wage": expected_wage24}).groupby("region").agg(
    n_wages=("wage", "count"), mean_wage=("wage", "mean"))
if isinstance(table1, pd.DataFrame) and table1.equals(expected_table):
    print("✓ Descriptive table correct")
else:
    print("Hint: group by normalized regions and count only valid wage entries.")

### ★★ Exercise 3 — construct a wage histogram

Create `hist_fig, hist_ax` using the cleaned 2024 wages, with x-axis `Annual wage (fictional USD)` and y-axis `People (count)`.

In [ ]:
hist_fig, hist_ax = None, None

In [ ]:
if (isinstance(hist_ax, plt.Axes) and len(hist_ax.patches) > 0
    and hist_ax.get_xlabel() == "Annual wage (fictional USD)"
    and hist_ax.get_ylabel() == "People (count)"):
    print("✓ Histogram constructed and labeled")
else:
    print("Hint: plot nonmissing wages and label both axes with units.")

### ★★★ Exercise 4 — plot regional average wages

Create `region_fig, region_ax` with a bar for each regional mean in `table1`, and label the y-axis with fictional USD units.

In [ ]:
region_fig, region_ax = None, None

In [ ]:
if (isinstance(region_ax, plt.Axes) and
    len(region_ax.patches) == len(expected_table) and
    region_ax.get_ylabel() == "Mean annual wage (fictional USD)"):
    print("✓ Regional comparison figure correct")
else:
    print("Hint: use ax.bar with table1's index and mean_wage column.")

### ★★★ Exercise 5 — plot average wage by wave

Create `trend_fig, trend_ax` plotting means for 2023 and 2024, after applying the same missing-value rules. Label both axes.

In [ ]:
trend_fig, trend_ax = None, None

In [ ]:
expected_trend = [pd.to_numeric(survey["wage_2023"], errors="coerce").replace(-99, np.nan).mean(),
                  expected_wage24.mean()]
if (isinstance(trend_ax, plt.Axes) and trend_ax.lines and
    np.allclose(trend_ax.lines[0].get_ydata(), expected_trend) and
    trend_ax.get_xlabel() and trend_ax.get_ylabel()):
    print("✓ Wave trend correct")
else:
    print("Hint: compute each wave mean after removing invalid wage values.")

## Common errors

- Mixing the original outlier-containing wage with cleaned values biases a descriptive mean.
- Grouping by unnormalized region labels creates artificial categories.
- Counting all rows instead of nonmissing wages overstates sample sizes.
- Graphing synthetic wages without identifying them as fictional invites mistaken interpretation.

## Recap

- Cleaning rules must be explicit and reused for every output.
- Table 1 should show valid sample counts and numerical units.
- A histogram, regional bar chart, and time-wave line chart answer distinct descriptive questions.
- A clean-kernel rerun is the basic reproducibility check.

**Self-assessment:** I can reproduce a Table 1 and all three figures from the fictional raw survey without manual data edits.

## Next steps

Continue to [Module 4 — Econometrics](../04_econometrics/README.md) when it is available. Review the [pandas groupby user guide](https://pandas.pydata.org/docs/user_guide/groupby.html). [Report a content issue](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md) with this notebook path.